In [15]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import random
from torchvision import transforms
from PIL import Image
import timm 
import ast
import importlib.util

DSL_PATH = 'dsl.py'  # Update this path as needed
CONSTANTS_PATH = 'constants.py'  # Update this path as needed

# Load the dsl module dynamically
spec = importlib.util.spec_from_file_location("dsl_module", DSL_PATH)
dsl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dsl_module)

# Load the constants module dynamically
spec_const = importlib.util.spec_from_file_location("constants_module", CONSTANTS_PATH)
constants_module = importlib.util.module_from_spec(spec_const)
spec_const.loader.exec_module(constants_module)

def parse_dsl_functions(dsl_path):
    """Parses the dsl.py file to extract functions with their argument and return types."""
    with open(dsl_path, 'r') as file:
        node = ast.parse(file.read(), filename=dsl_path)

    functions = []
    for n in node.body:
        if isinstance(n, ast.FunctionDef):
            func_name = n.name
            args = []
            # Extract argument names and types
            for arg in n.args.args:
                arg_name = arg.arg
                arg_type = None
                if arg.annotation:
                    arg_type = ast.unparse(arg.annotation)
                args.append((arg_name, arg_type))

            # Extract return type
            return_type = None
            if n.returns:
                return_type = ast.unparse(n.returns)

            # Get the function object from the module
            func = getattr(dsl_module, func_name, None)
            if func is not None:
                functions.append({
                    'name': func_name,
                    'function': func,
                    'args': args,
                    'return_type': return_type
                })
    return functions

dsl_functions = parse_dsl_functions(DSL_PATH)

In [23]:
constants_dict = {name: getattr(constants_module, name) for name in dir(constants_module) if not name.startswith('__')}

type_to_constants = {
    'Boolean': [True, False],
    'Integer': [v for v in constants_dict.values() if isinstance(v, int)],
    'IntegerTuple': [v for v in constants_dict.values() if isinstance(v, tuple) and all(isinstance(i, int) for i in v)],
}

In [291]:
def get_num_dimensions(t):
    if isinstance(t, tuple) and len(t) > 0:
        return 1 + get_num_dimensions(t[0])
    return 0
def generate_sample(upperweight=1.2, shrinkprob=0.05):
    # Function to convert a numpy array to a tuple of tuples
    def to_tuple_of_tuples(grid):
        return tuple(tuple(row) for row in grid)

    # Generate numbers for grid dimensions
    numbers = np.arange(2, 31)
    weights = upperweight ** numbers
    probabilities = weights / weights.sum()

    PRE_width = np.random.choice(numbers, p=probabilities)
    PRE_length = np.random.choice(numbers, p=probabilities)

    # Generate PRE grid using integers from 0 to 10
    PRE = np.random.randint(0, 11, size=(PRE_width, PRE_length), dtype=int)

    # Shrink the grid to a central rectangle with some probability
    if np.random.rand() < shrinkprob:
        new_width = np.random.randint(1, PRE_width + 1)
        new_length = np.random.randint(1, PRE_length + 1)
        row_start = (PRE_width - new_width) // 2
        col_start = (PRE_length - new_length) // 2
        
        # Create a new grid with zeros and place the smaller grid in the center
        new_PRE = np.zeros((PRE_width, PRE_length), dtype=int)
        new_PRE[row_start:row_start+new_width, col_start:col_start+new_length] = \
            np.random.randint(0, 11, size=(new_width, new_length))
        PRE = new_PRE

    # Convert PRE from numpy array to a tuple of tuples
    PRE = to_tuple_of_tuples(PRE)

    # Store PRE in variables and initialize grid as a copy of PRE
    variables = {'PRE': PRE}
    variable_types = {'PRE': 'Grid'}
    grid = PRE
    n_values = np.arange(1, 51)
    probabilities = n_values / n_values.sum()  # Probability proportional to n
    minimum_num_transformations = np.random.choice(n_values, p=probabilities)

    transformations_done = 0
    while True:
        func_info = random.choice(dsl_functions)
        func_name = func_info['name']
        func = func_info['function']
        arg_info = func_info['args']
        return_type = func_info['return_type']

        # Get arguments for the function
        args = []
        for arg_name, arg_type in arg_info:
            arg_value = None
            matching_vars = [variables[var_name] for var_name, var_type in variable_types.items() if var_type == arg_type]
            possible_values = type_to_constants.get(arg_type, [])
            combined_values = matching_vars + possible_values
            if combined_values:
                arg_value = random.choice(combined_values)
            if arg_value is None:
                continue
            args.append(arg_value)
        try:
            output = func(*args)
            # After minimum number of transformations, check if we can end
            if transformations_done >= minimum_num_transformations:
                if return_type == 'Grid':
                    grid = output
                    transformations_done += 1
                    break
                else:
                    variables[f'var_{transformations_done}'] = output
                    variable_types[f'var_{transformations_done}'] = return_type
                    transformations_done += 1
            else:
                variables[f'var_{transformations_done}'] = output
                variable_types[f'var_{transformations_done}'] = return_type
                transformations_done += 1
        except Exception as e:
            continue

        # Check if we have reached the desired number of transformations and can end
        if transformations_done >= minimum_num_transformations and return_type == 'Grid':
            break
    POST = grid

    if(get_num_dimensions(POST) < 2):
        return generate_sample()
    
    # Ensure homogenous shape. [I'm not sure why I need this].
    def ensure_homogeneous_shape(grid):
        max_length = max(len(row) for row in grid)
        return tuple(tuple(row + (11,) * (max_length - len(row))) for row in grid)
    POST = ensure_homogeneous_shape(POST)

    # Ensure POST is a valid grid by checking dimensions and applying pooling if necessary
    if len(POST) > 30 or len(POST[0]) > 30:
        # Apply pooling to reduce the dimensions to <= 30 by taking the upper left corner of each subrectangle
        def upper_left_pooling(grid, max_dim=30):
            grid_np = np.array(grid)
            if grid_np.shape[0] > max_dim:
                split_size = int(np.ceil(grid_np.shape[0] / max_dim))
                grid_np = grid_np[::split_size, :]
            if grid_np.shape[1] > max_dim:
                split_size = int(np.ceil(grid_np.shape[1] / max_dim))
                grid_np = grid_np[:, ::split_size]
            return to_tuple_of_tuples(grid_np)
        POST = upper_left_pooling(POST)
        transformations_done += 1

    # Pad PRE and POST to 30x30 with 11s (convert them to numpy for padding)
    PRE_np = np.array(PRE, dtype=int)
    POST_np = np.array(POST, dtype=int)

    PRE_np = np.pad(PRE_np, ((0, 30 - PRE_np.shape[0]), (0, 30 - PRE_np.shape[1])), mode='constant', constant_values=11)
    POST_np = np.pad(POST_np, ((0, 30 - POST_np.shape[0]), (0, 30 - POST_np.shape[1])), mode='constant', constant_values=11)

    # Concatenate PRE and POST to form a 30x60 grid
    sample_grid_np = np.hstack((PRE_np, POST_np))
    sample_grid = to_tuple_of_tuples(sample_grid_np)

    # Label is the number of transformations applied
    label = transformations_done
    return sample_grid, label


In [ ]:
class ARCDataset(Dataset):
    def __init__(self, num_samples):
        self.samples = []
        self.labels = []
        for _ in range(num_samples):
            sample, label = generate_sample()
            self.samples.append(sample)
            self.labels.append(label)
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        # Get the sample and label
        sample = self.samples[idx]
        label = self.labels[idx]
        
        # Normalize sample to [0, 255] and convert to uint8
        sample = np.clip((np.array(sample) / 10.0 * 255), 0, 255).astype(np.uint8)
        
        # Convert to PIL Image
        image = Image.fromarray(sample, mode='L')  # Grayscale image
        
        # Convert to RGB
        image = image.convert('RGB')
        
        # Apply transforms
        sample = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
        return sample, label

def create_vit_model():
    # Create a ViT model for 224x224 images
    model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=1)
    return model

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create dataset and data loader
train_dataset = ARCDataset(num_samples=10000)  # Adjust num_samples as needed
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Create model and move to device
model = create_vit_model().to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
num_epochs = 20  # Adjust the number of epochs as needed
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        # Move inputs and labels to device
        inputs = inputs.to(device)
        labels = labels.to(device).unsqueeze(1)  # Shape: (batch_size, 1)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')